# LightGBM with Fixed Data

This notebook applies classical machine learning to the problem using LightGBM decision trees. Separate models are trained for each of the 8 target features, representing horizons of 1, 3, 6 and 12 periods (1 period = 4x weeks/28 days). 

Models are saved to disk as pickle objects for loading. 

Note that the current models are "not great," they have a tendency to misclassify the positive "conflict" class due to the extreme minority of this class over the whole dataset. The sampling strategy should be revisited as the existing one resists the splitting of the id series (i.e, if an id is selected for the subsample, all observations in that observation are used). That was because it was adapted from an earlier notebook.

The definition of conflict remains localTotalFatalitis >= 3 OR localAvgSeverity >=3, same with regional conflict.

Local is defined as events assigned to place x + those within 10KM. Regional is defined as all events 10-50 Km from place x. 

## Load datasets

In [1]:
import pandas as pd

df_train = pd.read_csv("Dataset Preprocessed\eastafrica3_2026-09-20_Test.csv")
df_test= pd.read_csv("Dataset Preprocessed\eastafrica3_2026-09-20_Train.csv")

In [2]:
df_test.head(5)

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag,"Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag",Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag,Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag,"Population, total [SP.POP.TOTL]_missing_flag","Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag",Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag,Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag,"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag",is_voting
0,South_Sudan_Aduel,Aduel,South Sudan,6.3948,29.7938,191.651290,261.778096,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Uganda_Kigoma,Kigoma,Uganda,-0.5348,30.1209,36.414790,288.135035,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,Uganda_Kitagata,Kitagata,Uganda,-0.6736,30.1540,48.589624,290.160533,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,Sudan_Gereida,Gereida,Sudan,11.2749,25.1436,103.472672,938.733567,2015-01-01,2015-01-29,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,Sudan_El_Geneina,El Geneina,Sudan,13.4413,22.4454,17.029229,1122.870921,2015-01-01,2015-01-29,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


### Merge the datasets: not useful to be separated for now. 


In [3]:
df = pd.concat([df_train, df_test], ignore_index=True, sort=False)
#delete separated dataframes for memory usage reasons. 
del df_train
del df_test

In [4]:
#Force Garbage collection here. It's like programming in 1999. 
import gc
gc.collect()

29

### Column Conversion

In [5]:
df['id'] = df["id"].astype("category")
df['name'] = df["name"].astype("category")
df['country'] = df["country"].astype("category")
#df['country_id'] = df['country'].cat.codes may need this if getting country from the full dataset in conflictQuery is a non-starter.

df['periodStart'] = pd.to_datetime(df["periodStart"], format = '%Y-%m-%d')
df['periodEnd'] = pd.to_datetime(df["periodEnd"], format = '%Y-%m-%d')

exclude = {"id", "name", "country", "periodStart", "periodEnd"}

for col in df.columns:
    if col not in exclude:
        df[col] = pd.to_numeric(df[col], errors="raise")



,country,country_id
0,Central African Republic,0
1,Chad,1
2,Egypt,2
3,Ethiopia,3
4,Kenya,4
5,Libya,5
6,Somalia,6
7,South Sudan,7
8,Sudan,8
9,Uganda,9


In [6]:
#Sorting by periodStart, then id
df = df.sort_values(by=["periodStart", "id"], ascending=True)

In [7]:
df.shape

(1886230, 68)

In [8]:
testing_period_count = 12
# Use unique periods, then sort
unique_periods = sorted(df['periodStart'].unique())
training_periods = unique_periods[:-testing_period_count] # all but the last 12
testing_periods  = unique_periods[-testing_period_count:] # last 12
print("Full Range: ", min(df["periodStart"]), " - ", max(df["periodStart"]))
print ("Training Range: ", min(training_periods), " - ", max(training_periods))
print("Testing Range: ", min(testing_periods), " - ", max(testing_periods)) 

Full Range:  2015-01-01 00:00:00  -  2025-07-31 00:00:00
Training Range:  2015-01-01 00:00:00  -  2024-08-29 00:00:00
Testing Range:  2024-09-26 00:00:00  -  2025-07-31 00:00:00


## New target variables

## Drop the conflict_indicator: it's not relevant here. 

In [9]:
df = df[df.columns.drop(['conflict_indicator'])]

In [10]:
#Method to generate target cols. Horizons are 1 period into the future, 3, 6, and 12 periods into the future. 
horizons=[1, 3, 6, 12]
def generate_future_conflict_targets(df, horizons, target_columns = []):
    # Ensure correct ordering inside each series
    df = df.sort_values(["id", "periodStart"])

    # Columns that define conflict
    local_cols = ["LocalTotalFatalities", "LocalAvgSeverity"]
    regional_cols = ["RegionalTotalFatalities", "RegionalAvgSeverity"]

    for h in horizons:
        # Shift forward into the FUTURE (t + h)
        shifted_local = df.groupby("id", observed = True)[local_cols].shift(-h)
        shifted_regional = df.groupby("id", observed = True)[regional_cols].shift(-h)

        # Binary conflict flags
        df[f"TargetConflictLocal_t+{h}"] = (shifted_local >= 3).any(axis=1)
        target_columns.append(f"TargetConflictLocal_t+{h}")
#df = df[f"TargetConflictLocal_t+{h}"].astype('category')
        
        
        df[f"TargetConflictRegional_t+{h}"] = (shifted_regional >= 3).any(axis=1)
        target_columns.append(f"TargetConflictRegional_t+{h}")
       # df = df[f"TargetConflictLocal_t+{h}"].astype('category')
        
    return df

target_columns = []
df = generate_future_conflict_targets(df, horizons, target_columns)


unique_ids = df['id'].unique



In [11]:
df.columns

Index(['id', 'name', 'country', 'latitude', 'longitude', 'minBorderDistanceKm',
       'minCapitalDistanceKm', 'periodStart', 'periodEnd', 'LocalEventCount',
       'LocalDistinctActorCount', 'LocalDistinctEventTypes',
       'LocalDistinctEventSubtypes', 'LocalTotalFatalities',
       'LocalAvgSeverity', 'LocalProtestersEventCount',
       'LocalStateForcesEventCount', 'LocalPoliticalMilitiaEventCount',
       'LocalIdentityMilitiaEventCount', 'LocalRebelGroupEventCount',
       'LocalRiotersEventCount', 'LocalCiviliansEventCount',
       'LocalOtherTypeEventCount', 'RegionalEventCount',
       'RegionalDistinctActorCount', 'RegionalDistinctEventTypes',
       'RegionalDistinctEventSubTypes', 'RegionalTotalFatalities',
       'RegionalAvgSeverity', 'RegionalProtestersEventCount',
       'RegionalStateForcesEventCount', 'RegionalPoliticalMilitiaEventCount',
       'RegionalIdentityMilitiaEventCount', 'RegionalCiviliansEventCount',
       'RegionalRebelGroupEventCount', 'RegionalRioters

In [12]:
target_columns

['TargetConflictLocal_t+1',
 'TargetConflictRegional_t+1',
 'TargetConflictLocal_t+3',
 'TargetConflictRegional_t+3',
 'TargetConflictLocal_t+6',
 'TargetConflictRegional_t+6',
 'TargetConflictLocal_t+12',
 'TargetConflictRegional_t+12']

In [13]:
for col in df.columns:
    if (col in target_columns):
        df[col] = df[col].astype('category')

In [14]:
unique_ids = df['id'].unique
unique_ids

<bound method Series.unique of 169355    Central_African_Republic_Aba
182925    Central_African_Republic_Aba
196495    Central_African_Republic_Aba
210065    Central_African_Republic_Aba
223635    Central_African_Republic_Aba
                      ...             
95670                     Uganda_Zombo
109240                    Uganda_Zombo
122810                    Uganda_Zombo
136380                    Uganda_Zombo
149950                    Uganda_Zombo
Name: id, Length: 1886230, dtype: category
Categories (13560, object): ['Central_African_Republic_Aba', 'Central_African_Republic_Abagba_2', 'Central_African_Republic_Abba', 'Central_African_Republic_Abba-Bogani', ..., 'Uganda_Zirobwe', 'Uganda_Zoka', 'Uganda_Zoka_Forest', 'Uganda_Zombo']>

In [15]:
# Get raw unique ID values
unique_ids = df["id"].astype(str).unique()
one_id = unique_ids[0]
one_series = df[df["id"].astype(str) == one_id]

one_series.head(5)

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag",is_voting,TargetConflictLocal_t+1,TargetConflictRegional_t+1,TargetConflictLocal_t+3,TargetConflictRegional_t+3,TargetConflictLocal_t+6,TargetConflictRegional_t+6,TargetConflictLocal_t+12,TargetConflictRegional_t+12
169355,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2015-01-01,2015-01-29,0,...,NaN,0,False,False,False,False,False,False,False,False
182925,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2015-01-29,2015-02-26,0,...,NaN,0,False,False,False,False,False,False,False,False
196495,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2015-02-26,2015-03-26,0,...,NaN,0,False,False,False,False,False,False,False,False
210065,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2015-03-26,2015-04-23,0,...,NaN,0,False,False,False,False,False,False,False,False
223635,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2015-04-23,2015-05-21,0,...,NaN,0,False,False,False,False,False,False,False,False


In [16]:
one_series.tail(5)

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,periodEnd,LocalEventCount,...,"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag",is_voting,TargetConflictLocal_t+1,TargetConflictRegional_t+1,TargetConflictLocal_t+3,TargetConflictRegional_t+3,TargetConflictLocal_t+6,TargetConflictRegional_t+6,TargetConflictLocal_t+12,TargetConflictRegional_t+12
101505,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2025-04-10,2025-05-08,0,...,1.0,0,False,False,False,False,False,False,False,False
115075,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2025-05-08,2025-06-05,0,...,1.0,0,False,False,False,False,False,False,False,False
128645,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2025-06-05,2025-07-03,0,...,1.0,0,False,False,False,False,False,False,False,False
142215,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2025-07-03,2025-07-31,0,...,1.0,0,False,False,False,False,False,False,False,False
155785,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.3149,61.576638,411.333393,2025-07-31,2025-08-28,0,...,1.0,0,False,False,False,False,False,False,False,False


## Generate ARIMA style lag, avg and volatility (std dev) features for the raw feature counts. 

In [17]:
acled_raw_features = ['LocalEventCount', 'LocalDistinctActorCount', 'LocalDistinctEventTypes',
       'LocalDistinctEventSubtypes', 'LocalTotalFatalities',
       'LocalAvgSeverity', 'LocalProtestersEventCount',
       'LocalStateForcesEventCount', 'LocalPoliticalMilitiaEventCount',
       'LocalIdentityMilitiaEventCount', 'LocalRebelGroupEventCount',
       'LocalRiotersEventCount', 'LocalCiviliansEventCount',
       'LocalOtherTypeEventCount', 'RegionalEventCount',
       'RegionalDistinctActorCount', 'RegionalDistinctEventTypes',
       'RegionalDistinctEventSubTypes', 'RegionalTotalFatalities',
       'RegionalAvgSeverity', 'RegionalProtestersEventCount',
       'RegionalStateForcesEventCount', 'RegionalPoliticalMilitiaEventCount',
       'RegionalIdentityMilitiaEventCount', 'RegionalCiviliansEventCount',
       'RegionalRebelGroupEventCount', 'RegionalRiotersEventCount',
       'RegionalOtherTypeEventCount']

country_facts_scaled_features = [
       'Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST]',
       'Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]',
       'Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]',
       'Children out of school (% of primary school age) [SE.PRM.UNER.ZS]',
       'GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]',
       'GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]',
       'Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS]',
       'Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]',
       'Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]',
       'Cereal yield (kg per hectare) [AG.YLD.CREL.KG]',
       'Permanent cropland (% of land area) [AG.LND.CROP.ZS]',
       'Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]'
]

country_facts_unscaled_features =[
 'Population, total [SP.POP.TOTL]',
 'Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]'
]

country_fact_missing_flags = ['Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS]_missing_flag',
       'Cereal yield (kg per hectare) [AG.YLD.CREL.KG]_missing_flag',
       'Children out of school (% of primary school age) [SE.PRM.UNER.ZS]_missing_flag',
       'Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST]_missing_flag',
       'Country Code_missing_flag',
       'GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]_missing_flag',
       'GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]_missing_flag',
       'Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]_missing_flag',
       'Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]_missing_flag',
       'Permanent cropland (% of land area) [AG.LND.CROP.ZS]_missing_flag',
       'Population, total [SP.POP.TOTL]_missing_flag',
       'Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]_missing_flag',
       'Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]_missing_flag',
       'Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]_missing_flag',
       'Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]_missing_flag'
]

covariates = ['country', 'latitude', 'longitude', 'minBorderDistanceKm', 'minCapitalDistanceKm', 'is_voting']

date_columns = ['periodStart', 'periodEnd']

arima_lags = [1, 2, 3, 6, 12, 18]
arima_windows = [1, 2, 3, 6, 12, 18]

### Period_midpoint: Between period_Start and period_end

In [18]:
df["period_midpoint"] = df["periodStart"] + (df["periodEnd"] - df["periodStart"]) / 2

### Month_sin and month_cos: Encoding month but keeping December close to January

In [19]:
import numpy as np
df["month"] = df["period_midpoint"].dt.month #temporary
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df = df[df.columns.drop(['month'])] #Remove the raw month: we don't care since the sin and cos variations capture thsi better.

### Apply ACLED Feature Lags

In [20]:

#raw values from the previous n period. 
def generate_lag_features(df, feature_cols, lags):
    # Ensure correct ordering inside each series
    df = df.sort_values(["id", "periodStart"])

    for col in feature_cols:
        for h in lags:
            df[f"{col}_lag{h}"] = df.groupby("id", observed = True)[col].shift(h)

    return df

#Rolling median over previous n periods. 
def generate_rolling_median_features(df, feature_cols, windows):
    # Ensure correct ordering inside each series
    df = df.sort_values(["id", "periodStart"])

    for col in feature_cols:
        for w in windows:
            df[f"{col}_median{w}"] = (
                df.groupby("id",  observed = True)[col]
                  .rolling(window=w, min_periods=1)
                  .median()
                  .reset_index(level=0, drop=True)
            )

    return df

#rolling std deviation (volatility over previous n periods
def generate_rolling_std_features(df, feature_cols, windows):
    # Ensure correct ordering inside each series
    df = df.sort_values(["id", "periodStart"])

    for col in feature_cols:
        for w in windows:
            df[f"{col}_std{w}"] = (
                df.groupby("id",  observed = True)[col]
                  .rolling(window=w, min_periods=1)
                  .std()
                  .reset_index(level=0, drop=True)
            )

    return df




In [21]:
startTime = pd.Timestamp.now()
df = generate_rolling_std_features(df, acled_raw_features, arima_windows) # 3 mins
print("generate_rolling_std_features: ", pd.Timestamp.now() - startTime)

C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_std{w}"] = (
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_std{w}"] = (
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.

generate_rolling_std_features:  0 days 00:02:49.887888


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_std{w}"] = (


In [22]:
startTime = pd.Timestamp.now()
df = generate_lag_features(df, acled_raw_features, arima_lags) #9 mins
print("generate_lag_features: ", pd.Timestamp.now() - startTime)
 

C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{h}"] = df.groupby("id", observed = True)[col].shift(h)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{h}"] = df.groupby("id", observed = True)[col].shift(h)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert

generate_lag_features:  0 days 00:00:16.237248


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{h}"] = df.groupby("id", observed = True)[col].shift(h)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{h}"] = df.groupby("id", observed = True)[col].shift(h)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert

In [23]:
startTime = pd.Timestamp.now()
df = generate_rolling_median_features(df, acled_raw_features, arima_windows) #5 mins
print("generate_rolling_median_features: ", pd.Timestamp.now() - startTime)

C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_median{w}"] = (
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_median{w}"] = (
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usi

generate_rolling_median_features:  0 days 00:04:27.636772


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\435574386.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_median{w}"] = (


In [24]:
df = df[df.columns.drop(acled_raw_features)] #drop the raw features, not required now.


### Possible Kernal Failure!
Note the kernal does like to fall over at this point, if the line generate_rolling_std_features:  0 days 00:12:28.653873 is output then it's been successful. The kernal can be interrupted and following cells run. 

### Scale and Apply Country_Fact Features:

In [25]:
def generate_unscaled_lag_features(df, features, lags):
   #Generate log 
    for feature in features:
        for lag in lags:

            #temporary, it's just the raw values. 
            lag_col = f"{feature}_t{lag}"
            log_col = f"{feature}_t{lag}_log" #rate of change. 
            
            #create lagged feature
            df[lag_col] = df.groupby("country")[feature].shift(lag)
            
            # Log transform the lagged feature to capture the rate of change:
            df[log_col] = np.log1p(df[lag_col])

            #delete the temporary column:
            df.drop(columns=[lag_col], inplace=True)
    
    return df

def generate_scaled_rate_features(df, features, lag_pair):
    #Applies rate of change columns as a percentage. Intended for the previous 13 and 26 lags. 

    lag1, lag2 = lag_pair

    index = 0
    for feature in features:
        index = index + 1
        # Temporary lag columns
        lag1_col = f"{feature}_t{lag1}"
        lag2_col = f"{feature}_t{lag2}"

        #create temportary lagged features
        df[lag1_col] = df.groupby("country")[feature].shift(lag1)
        df[lag2_col] = df.groupby("country")[feature].shift(lag2)

        #rate of change between t-lag1 and t-lag2
        rate_col = f"{feature}_rate_t{lag1}_t{lag2}"
        df[rate_col] = (df[lag1_col] - df[lag2_col]) / df[lag2_col]

        #delete temporary lag columns
        df.drop(columns=[lag1_col, lag2_col], inplace=True)

        print(f"done feature {index} of {len(features)}")
    return df

In [26]:
gc.collect()
df = df.sort_values(["country", "periodStart"])

In [27]:
#Apply rate of change to the scaled features:
startTime = pd.Timestamp.now()
df = generate_scaled_rate_features(df, country_facts_scaled_features, [13, 26]) #1 year, 2 years. 
print("generate_scaled_rate_features: ", pd.Timestamp.now() - startTime) #takes ~5 mins. If it runs: likes to fail on RAM usage. 

C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior o

done feature 1 of 12


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior o

done feature 2 of 12


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior o

...

done feature 10 of 12


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior o

done feature 11 of 12


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag1_col] = df.groupby("country")[feature].shift(lag1)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior o

done feature 12 of 12
generate_scaled_rate_features:  0 days 00:03:34.754484


In [28]:
#scale (log) and apply rate-of-change to the unscaled features
startTime = pd.Timestamp.now()
df = generate_unscaled_lag_features(df, country_facts_unscaled_features, [13, 26]) #1 year, 2 years. 
print("generate_unscaled_lag_features: ", pd.Timestamp.now() - startTime) #47s 

C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[lag_col] = df.groupby("country")[feature].shift(lag)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lag_col] = df.groupby("country")[feature].shift(lag)
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\254497565.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all co

generate_unscaled_lag_features:  0 days 00:00:53.054643


In [29]:
df = df[df.columns.drop(country_facts_unscaled_features)] #drop the raw features, not required now.
df = df[df.columns.drop(country_facts_scaled_features)]

### Drop raw columns:

In [30]:
df = df[df.columns.drop(country_fact_missing_flags)]

### Rename all columns to remove special chars

In [31]:
import re
new_cols = {}

for col in df.columns:
    new_name = col.replace(" ", "_") #Replace spaces with underscores
    new_name = re.sub(r"[^A-Za-z0-9_+]", "", new_name) #remove any character that is not A-Z, a-z, 0-9, or underscore or +. + included as it's already in the target_columns. 
    new_cols[col] = new_name

df = df.rename(columns=new_cols)

In [32]:
#print(df.columns)
print(df.shape)
print("Columns: ")
for col in df.columns:
    print(col)


(1886230, 541)
Columns: 
id
name
country
latitude
longitude
minBorderDistanceKm
minCapitalDistanceKm
periodStart
periodEnd
is_voting
TargetConflictLocal_t+1
TargetConflictRegional_t+1
TargetConflictLocal_t+3
TargetConflictRegional_t+3
TargetConflictLocal_t+6
TargetConflictRegional_t+6
TargetConflictLocal_t+12
TargetConflictRegional_t+12
period_midpoint
month_sin
month_cos
LocalEventCount_std1
LocalEventCount_std2
LocalEventCount_std3
LocalEventCount_std6
LocalEventCount_std12
LocalEventCount_std18
LocalDistinctActorCount_std1
LocalDistinctActorCount_std2
LocalDistinctActorCount_std3
LocalDistinctActorCount_std6
LocalDistinctActorCount_std12
LocalDistinctActorCount_std18
LocalDistinctEventTypes_std1
LocalDistinctEventTypes_std2
LocalDistinctEventTypes_std3
LocalDistinctEventTypes_std6
LocalDistinctEventTypes_std12
LocalDistinctEventTypes_std18
LocalDistinctEventSubtypes_std1
LocalDistinctEventSubtypes_std2
LocalDistinctEventSubtypes_std3
LocalDistinctEventSubtypes_std6
LocalDistinctEven

### Export the dataset as .parquet for loading in ConflictQuery

In [33]:
#Save the whole dataframe to parquet. We'll use this as the datasource when making predictions in the API. 
df.to_parquet("Dataset Preprocessed/eastafrica_preprocessed_for_LightGBM.parquet")

## Training LightGBM

In [34]:
gc.collect() #force garbage collection prior to training. 

0

In [35]:
print("target_columns: " ,target_columns)

target_columns:  ['TargetConflictLocal_t+1', 'TargetConflictRegional_t+1', 'TargetConflictLocal_t+3', 'TargetConflictRegional_t+3', 'TargetConflictLocal_t+6', 'TargetConflictRegional_t+6', 'TargetConflictLocal_t+12', 'TargetConflictRegional_t+12']


In [36]:
models = {}  # dictionary to store one model per target

In [37]:
testing_period_count = 12
unique_periods = sorted(df['periodStart'].unique())

training_periods = unique_periods[:-testing_period_count]
testing_periods  = unique_periods[-testing_period_count:]


In [38]:
exclude_columns = set(target_columns)
exclude_columns.update([
    "periodStart",     # time index
    'period_midpoint' # LigtGBm is not time aware. 
    'periodEnd'
    'name',
    #'country' #get rid of country, keep country_id. This means that the categorical encoding is preserved in the data, and can be 
])
feature_columns = [
    col for col in df.columns
    if col not in exclude_columns
    and (df[col].dtype.kind in "biufc" or df[col].dtype == "category")
]

df[target_columns] = df[target_columns].apply(pd.to_numeric, errors="raise").astype("Int64")


## Sampling Strategy

In [39]:
subsamples_train = {}
subsamples_test = {}

# Identify last 12 periods
last_12_periods = (
    df["periodStart"]
    .sort_values()
    .unique()[-12:]
)

# IDs that appear in last 12 periods
ids_in_last_12 = df[df["periodStart"].isin(last_12_periods)]["id"].unique().tolist()
ids_not_in_last_12 = df[~df["periodStart"].isin(last_12_periods)]["id"].unique().tolist()

for target_column in target_columns:

    # Find IDs with positives IN the last 12 periods
    positive_in_last_12 = (
        df[df["periodStart"].isin(last_12_periods)]
        .groupby("id")[target_column]
        .max()
    )
    positive_in_last_12 = positive_in_last_12[positive_in_last_12 == 1].index.tolist()

    # Find IDs with positives OUTSIDE the last 12 periods
    positive_outside_last_12 = (
        df[~df["periodStart"].isin(last_12_periods)]
        .groupby("id")[target_column]
        .max()
    )
    positive_outside_last_12 = positive_outside_last_12[positive_outside_last_12 == 1].index.tolist()

    # Negative IDs (no positives anywhere)
    id_max = df.groupby("id")[target_column].max()
    negative_ids = id_max[id_max == 0].index.tolist()

    # Sample negatives separately for train/test
    np.random.seed(1)

    negative_train = [i for i in negative_ids if i in ids_not_in_last_12]
    negative_test  = [i for i in negative_ids if i in ids_in_last_12]

    sampled_negative_train = np.random.choice(negative_train, size=min(150, len(negative_train)), replace=False)
    sampled_negative_test  = np.random.choice(negative_test,  size=min(150, len(negative_test)),  replace=False)

    # Final ID lists
    train_ids = list(positive_outside_last_12) + list(sampled_negative_train)
    test_ids  = list(positive_in_last_12)      + list(sampled_negative_test)

    df_train = df[df["id"].isin(train_ids)]
    df_test  = df[df["id"].isin(test_ids)]

    print(f"\n{target_column}")
    print("Train positives:", df_train[target_column].sum())
    print("Test positives:", df_test[target_column].sum())

    subsamples_train[target_column] = train_ids
    subsamples_test[target_column]  = test_ids


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  id_max = df.groupby("i


TargetConflictLocal_t+1
Train positives: 35881
Test positives: 21617


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  id_max = df.groupby("i


TargetConflictRegional_t+1
Train positives: 2361
Test positives: 1640


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  id_max = df.groupby("i

...


TargetConflictRegional_t+6
Train positives: 2274
Test positives: 1410


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  id_max = df.groupby("i


TargetConflictLocal_t+12
Train positives: 34845
Test positives: 6


C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("id")[target_column]
C:\Users\andre\AppData\Local\Temp\ipykernel_11836\2129369497.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  id_max = df.groupby("i


TargetConflictRegional_t+12
Train positives: 2181
Test positives: 0


### Training
Not sure the below is entirely right, is X_train being applied to all models with all target columns?

In [47]:
#Documentation: https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html
import lightgbm as lgb
import joblib
models = {}

for target_column in target_columns:

    gc.collect()
    
    startTime = pd.Timestamp.now()
    print(f"training for {target_column} started at: ", startTime)

    # IDs assigned to train for this target
    train_ids = subsamples_train[target_column]

    # Extract only the rows for these IDs
    df_train = df[df["id"].isin(train_ids)].copy()

    # Build y_train as a dataframe (one column)
    y_train = df_train[[target_column]].copy()

    # Build X_train as a dataframe (drop target + ID + raw datetime columns)
    feature_cols = [
        c for c in df_train.columns
        if c not in target_columns
        and c not in ["id", 'name',"periodStart", "periodEnd", "period_midpoint"]
    ]

    X_train = df_train[feature_cols].copy()

    #We need to output the categorical encoding to ensure we can force it in ConflictQuery. 
    country_mapping_df = pd.DataFrame({
    'country': df['country'].cat.categories,
    'country_id': range(len(df['country'].cat.categories))
    })
    country_mapping_df.to_csv(f"lightgbm_country_encoding.csv", index=False, mode='w+') #I tested this, it's the same values for each model. 

    #Useful for checking what we actually trained on:
    X_train.head(5).to_csv(f"lightgbm_X_train_{target_column}.csv", index=False, mode='w+')
    y_train.head(5).to_csv(f"lightgbm_y_train_{target_column}.csv", index=False, mode='w+')
    
    print("X_train shape:", X_train.shape)
    print("y_train positives:", y_train[target_column].sum())

    # ---- Train your model here ----
    model = lgb.LGBMClassifier(
        boosting_type='gbdt',
        objective="binary",
        learning_rate=0.005,#num_leaves=31,
        n_estimators=500, #default 100 just one for now to validate the training and evaluation process. 
        #is_unbalance=True #leave this ou for now- let the model assume balanced classes. 
        random_state=1,
        n_jobs=5, #My intel i7-12700H has six cores. Setting this to six is *very* bad idea and crashes... everything. 
        #eval_metric="binary_logloss" #it doesnnt know what this is, even though it's specified in the documentation. 
       # verbose = -1
    )

    #initiate training:
    model.fit(X_train, y_train[target_column])

    #Save the model to disk as a pickle file. 
    #model.booster_.save_model(f"lightgbm_model_{target}.txt")
    joblib.dump(model, f"lightgbm_model_{target_column}.pkl") #save the full model as a pickle file. 
    
    #to load:
    #joblib.load(f"lightgbm_model_{target_column}.pkl")
    
    models[target_column] = model
    
    # ---- Free memory ----
    del df_train
    del X_train
    del y_train
    gc.collect()

    print(f"Duration for {target_column}: ", pd.Timestamp.now() - startTime)
    print("------")

training for TargetConflictLocal_t+1 started at:  2026-09-26 16:37:14.973962
X_train shape: (1097127, 528)
y_train positives: 35881
[LightGBM] [Info] Number of positive: 35881, number of negative: 1061246
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.635617 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 36818
[LightGBM] [Info] Number of data points in the train set: 1097127, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.032705 -> initscore=-3.386991
[LightGBM] [Info] Start training from score -3.386991
Duration for TargetConflictLocal_t+1:  0 days 00:01:05.578643
------
training for TargetConflictRegional_t+1 started at:  2026-09-26 16:38:20.646911
X_train shape: (56156, 528)
y_train positives: 2361
[LightGBM] [Info] Number of positive: 2361, number of negative: 53795
[LightGBM] [Info] Auto-choosing r

### Calculate Evaluation Metrics:

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Model Evaluation:")

for target_column in target_columns:

    print(f"\nEvaluating model for: {target_column}")

    #build testing dataset (y; actual target values)
    test_ids = subsamples_test[target_column]
    df_test = df[df["id"].isin(test_ids)].copy()

    y_test = df_test[[target_column]].copy()

    #build the testing dataset (x):
    feature_cols = [
        c for c in df_test.columns
        if c not in target_columns
        and c not in ["id", "placeName", "periodStart", "periodEnd", "period_midpoint"]
    ]

    X_test = df_test[feature_cols].copy()

    print("X_test shape:", X_test.shape)
    print("y_test positives:", y_test[target_column].sum())

    #obtain model from dict
    #model = models[target_column]

    #load model from disk (pickle files are neat, this saves tedious mucking around with only saving part of the model):
    model = joblib.load(f"lightgbm_model_{target_column}.pkl")
    
    #predict
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_prob >= 0.5).astype(int)

    #Conf matrix
    cm = confusion_matrix(y_test, y_pred) #note this all breaks for regional_12, as there are 0 positive instances. 
    tn, fp, fn, tp = cm.ravel()

    #specificity = TN / (TN + FP)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    #Classification metrics:
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_pred_prob) if y_test[target_column].nunique() > 1 else float("nan")

    print("\nMetrics:")
    print(f"Accuracy:      {accuracy:.4f}")
    print(f"Precision:     {precision:.4f}")
    print(f"Recall:        {recall:.4f}")
    print(f"Specificity:   {specificity:.4f}")
    print(f"F1 Score:      {f1:.4f}")
    print(f"AUC:           {auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(cm)

    print("\nFeature Importance (by gain):")
    importance = model.feature_importances_
    #importance = model.feature_importance(importance_type='gain')

    
    fi_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": importance
    }).sort_values("importance", ascending=False)

    print(fi_df.head(20))  # top 20 features
    
    #Delete iteration variables to free up memory:
    del df_test, X_test, y_test, y_pred, y_pred_prob, cm, fi_df
    gc.collect()

    print(f"End of evaluation for model {target_column}")
    print("------")
    #end of this iteration
    

Note that all models have a tendancy to classify Conflict as non-conflict across all targets. Is the model useless? Yes and no. When the models predict no conflict, they are strong. Conflict predictions need to be taken with a (large) grain of salt. Why is this happening? Conflict examples (positive class) is very much the minority, especially for the regional (10-50KM) models. Ideally we'd give the models equal numbers of the classes as examples, but some target features are so extremely minority that this is not practial. So need to adjust the sampling strategy further. Since the LightGBM model is NOT time aware, it is acceptable to split the ids, which we have avoided up to now. 

## Future actions:
* validate all models will train (they do!)
* analyse performance (done, to a degree, TargetConflictRegional_t+12 fails to evaluate as there are missing positive cases)
* apply stratification (done, to a degree, this is something that can be revised: positive classes are still the minority amongst all models' training sets. )
* load the country facts features + retrain (done)
* work out how to use preds and probs. (done)

AttributeError: module 'gc' has no attribute 'delete'